# Séance 9 — Mini-projet CNN

**Objectif.** Aucune notion nouvelle cette séance : vous mobilisez ce qui a été vu aux
séances 7 et 8 (architecture, blocs résiduels, batch normalization) et aux séances 4-5
(loss, initialisation, dropout, weight decay, early stopping) sur un problème un cran plus
réaliste. Entraînement **from scratch**, sans modèle pré-entraîné (ça, c'est réservé à la
séance 11).

**Consignes.** Toute la plomberie (données, dataloaders, boucle d'entraînement, visualisation)
est fournie ci-dessous et fonctionne telle quelle. Le travail porte sur les **choix de
modélisation** :

- architecture (profondeur, largeur, éventuellement blocs résiduels) ;
- régularisation (dropout, weight decay, batch normalization, early stopping) ;
- optimiseur (SGD, SGD+momentum, Adam...) et ses hyperparamètres ;
- data augmentation (`torchvision.transforms`).

Vous devez comparer **au moins 3 configurations** et justifier vos choix à partir des courbes
d'apprentissage (train vs val), pas seulement du score final.

**Données.** Un sous-ensemble de CIFAR-10 (images couleur 32x32, 10 classes), volontairement
réduit pour que les entraînements restent raisonnables sur CPU en séance.


In [ ]:
# !pip install -q torch torchvision matplotlib

import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from training_toolbox import Trainer, EarlyStopping, ModelCheckpoint, accuracy

torch.manual_seed(0)


## Partie 0 — Données (plomberie fournie)

On sous-échantillonne CIFAR-10 (1 image sur 4 pour l'entraînement) pour garder des temps
d'entraînement compatibles avec une séance de TP sur CPU. Deux jeux de transforms sont
fournis : un sans augmentation, un avec (recadrage aléatoire + flip horizontal, classiques
sur des images naturelles).


In [ ]:
CLASSES = [
    "avion", "voiture", "oiseau", "chat", "cerf",
    "chien", "grenouille", "cheval", "bateau", "camion",
]

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

transform_plain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

transform_augmented = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Deux instances du même dataset, avec des transforms différents : pratique pour basculer
# facilement de l'un à l'autre sans redéfinir le sous-échantillonnage.
train_full_plain = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_plain
)
train_full_augmented = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_augmented
)
test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_plain
)

# Sous-échantillonnage : 1 exemple sur 4, indices partagés entre train/val
indices = list(range(0, len(train_full_plain), 4))
n_val = len(indices) // 5
val_indices, train_indices = indices[:n_val], indices[n_val:]

val_set = Subset(train_full_plain, val_indices)          # jamais augmenté
train_set_plain = Subset(train_full_plain, train_indices)
train_set_augmented = Subset(train_full_augmented, train_indices)

print(f"Train : {len(train_indices)} images - Val : {len(val_indices)} images "
      f"- Test : {len(test_set)} images")

val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)


def make_train_loader(use_augmentation=False, batch_size=128):
    dataset = train_set_augmented if use_augmentation else train_set_plain
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [ ]:
def unnormalize(img):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1)

x_batch, y_batch = next(iter(make_train_loader(use_augmentation=True)))
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, img, label in zip(axes.ravel(), x_batch, y_batch):
    ax.imshow(unnormalize(img).permute(1, 2, 0))
    ax.set_title(CLASSES[label.item()], fontsize=9)
    ax.axis("off")
plt.suptitle("Exemples avec data augmentation")
plt.tight_layout()
plt.show()


## Partie 1 — Un point de départ : architecture configurable

Pour ne pas repartir de zéro sur la définition du modèle (déjà travaillée séances 7-8), on
vous fournit une architecture **configurable**, à ajuster selon les configurations que vous
voulez comparer : nombre de blocs conv, présence de batch norm, taux de dropout avant le
classifieur.


In [ ]:
class ConfigurableCNN(nn.Module):
    """CNN mini-VGG configurable : `n_blocks` blocs (conv 3x3, BN optionnelle, ReLU,
    MaxPool2d(2)), le nombre de canaux doublant à chaque bloc, puis un classifieur dense
    avec dropout optionnel.
    """

    def __init__(self, n_blocks=3, base_channels=32, use_batchnorm=True,
                 dropout=0.3, n_classes=10):
        super().__init__()
        layers = []
        in_c = 3
        out_c = base_channels
        for _ in range(n_blocks):
            layers.append(nn.Conv2d(in_c, out_c, kernel_size=3, padding=1,
                                     bias=not use_batchnorm))
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2))
            in_c, out_c = out_c, out_c * 2
        self.features = nn.Sequential(*layers)

        spatial = 32 // (2 ** n_blocks)
        flat_dim = in_c * spatial * spatial
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(flat_dim, 128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Sanity check
model = ConfigurableCNN(n_blocks=3, use_batchnorm=True, dropout=0.3)
with torch.no_grad():
    out = model(x_batch)
print("Sortie :", out.shape)
print(f"Paramètres : {sum(p.numel() for p in model.parameters()):,}")


## Partie 2 — À vous de jouer : comparez plusieurs configurations

Complétez le dictionnaire `configs` ci-dessous avec **au moins 3 configurations** que vous
voulez comparer. Une configuration précise :

- les arguments de `ConfigurableCNN` (architecture) ;
- `use_augmentation` (data augmentation ou non) ;
- l'optimiseur à construire (SGD, SGD+momentum, Adam...) et son learning rate / weight decay ;
- éventuellement des callbacks (`EarlyStopping`, `ModelCheckpoint`).

Quelques pistes de comparaison possibles (choisissez-en 3, ou plus si le temps le permet) :

- Adam vs SGD+momentum à architecture fixée.
- avec/sans data augmentation, à architecture et optimiseur fixés.
- avec/sans dropout (ou différents taux), à architecture fixée.
- `n_blocks=2` vs `n_blocks=3` (profondeur) à budget d'epochs identique.
- avec/sans weight decay (`torch.optim.Adam(..., weight_decay=...)`).

**TODO : complétez `configs`.**


In [ ]:
def build_optimizer(model, kind="adam", lr=1e-3, weight_decay=0.0, momentum=0.9):
    if kind == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif kind == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum,
                                weight_decay=weight_decay)
    raise ValueError(kind)


# TODO : ajustez / complétez ces configurations (au moins 3), par exemple :
configs = {
    "baseline_adam": dict(
        model_kwargs=dict(n_blocks=3, use_batchnorm=True, dropout=0.3),
        use_augmentation=False,
        optimizer_kwargs=dict(kind="adam", lr=1e-3),
    ),
    # TODO : une variante avec data augmentation
    "augmentation": dict(
        model_kwargs=dict(n_blocks=3, use_batchnorm=True, dropout=0.3),
        use_augmentation=None,  # <- à fixer (True)
        optimizer_kwargs=dict(kind="adam", lr=1e-3),
    ),
    # TODO : une variante avec un autre optimiseur (et/ou du weight decay)
    "sgd_momentum": dict(
        model_kwargs=dict(n_blocks=3, use_batchnorm=True, dropout=0.3),
        use_augmentation=False,
        optimizer_kwargs=None,  # <- à fixer (kind="sgd", lr=..., momentum=..., weight_decay=...)
    ),
    # TODO : ajoutez vos propres configurations ici
}


In [ ]:
results = {}

for name, cfg in configs.items():
    print(f"\n=== Configuration : {name} ===")
    model = ConfigurableCNN(**cfg["model_kwargs"])
    optimizer = build_optimizer(model, **cfg["optimizer_kwargs"])
    loss_fn = nn.CrossEntropyLoss()

    callbacks = [EarlyStopping(patience=4, monitor="val_loss")]
    trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy},
                       callbacks=callbacks)

    train_loader = make_train_loader(use_augmentation=cfg["use_augmentation"])

    t0 = time.time()
    history = trainer.fit(train_loader, val_loader, epochs=15, verbose=False)
    elapsed = time.time() - t0

    test_stats = trainer.evaluate(test_loader)
    results[name] = dict(history=history, test=test_stats, time=elapsed)
    print(f"  temps: {elapsed:.1f}s - epochs réellement effectuées: "
          f"{len(history['train_loss'])} - test acc: {test_stats['acc']:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, res in results.items():
    axes[0].plot(res["history"]["val_loss"], label=name)
    axes[1].plot(res["history"]["val_acc"], label=name)
axes[0].set_title("Validation loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"{'config':<20}{'test acc':>10}{'temps (s)':>12}")
for name, res in results.items():
    print(f"{name:<20}{res['test']['acc']:>10.4f}{res['time']:>12.1f}")


## Synthèse (à rédiger)

- Quelle configuration donne la meilleure accuracy de test ? Est-ce aussi celle qui
  généralise le mieux (écart train/val le plus faible), ou juste celle qui overfit le moins
  vite en 15 epochs ?
- La data augmentation a-t-elle un effet visible sur l'écart train/val ? Sur la vitesse de
  convergence par epoch (attention : une epoch avec augmentation "voit" des versions
  différentes des mêmes images, ce n'est pas strictement comparable à une epoch sans
  augmentation) ?
- Le choix d'optimiseur change-t-il la vitesse de convergence ? Le score final ?
- Si vous deviez choisir une seule configuration pour un déploiement réel, laquelle
  choisiriez-vous, et sur quels critères (score, temps d'entraînement, stabilité) ?

## Pour aller plus loin (optionnel)

- Réutiliser le `ResidualBlock` de la séance 8 dans `ConfigurableCNN` pour comparer une
  version résiduelle à profondeur comparable.
- Recherche d'hyperparamètres plus systématique (grid/random search sur `dropout`, `lr`,
  `n_blocks`), comme en séance 6.
- Entraîner plus longtemps (plus d'epochs, patience d'`EarlyStopping` plus grande) hors
  séance, sur l'intégralité de CIFAR-10, pour voir si les conclusions tiennent à plus grande
  échelle.
